In [1]:
import gc
import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
BASE_DIR = Path(
    r"F:\Mayank\ml-research\ml-relativistic-simulator"
)

MODEL_DIR = BASE_DIR / "models"
TEST_FILE = BASE_DIR / "data" / "relativistic_extreme_test_10k.npy"

RESULT_FILE = (
    BASE_DIR / "data" / "extreme_test_results.csv"
)

In [3]:
target_names = [
    "position",
    "velocity",
    "beta",
    "gamma",
    "momentum",
    "kinetic_energy",
    "total_energy",
    "proper_time",
]

In [4]:
model_files = [
    "linear_regression.joblib",
    "polynomial_regression_degree_2.joblib",
    "polynomial_regression_degree_3.joblib",
    "polynomial_regression_degree_4.joblib",
    "decision_tree.joblib",
    "gradient_boosting.joblib",
    "hist_gradient_boosting.joblib",
    "ExtraTreesRegressor.joblib",
    "RandomForestRegressor.joblib",
]

In [6]:
print("EXTREME TEST SET EVALUATION")
data = np.load(TEST_FILE)
print(f"Dataset shape : {data.shape}")
print(f"Dataset dtype : {data.dtype}")
print(f"Dataset memory: {data.nbytes / 1024**2:.2f} MB")

EXTREME TEST SET EVALUATION
Dataset shape : (10000, 13)
Dataset dtype : float64
Dataset memory: 0.99 MB


In [7]:
# Inputs:
# 0 = mass
# 1 = force
# 2 = initial_velocity
# 3 = initial_beta
# 4 = time

X_test = data[:, [0, 1, 2, 3, 4]]

In [8]:
# Outputs:
# 5  = position
# 6  = velocity
# 7  = beta
# 8  = gamma
# 9  = momentum
# 10 = kinetic_energy
# 11 = total_energy
# 12 = proper_time

Y_test = data[:, 5:13]

In [9]:
print("\nX_test:", X_test.shape)
print("Y_test:", Y_test.shape)


X_test: (10000, 5)
Y_test: (10000, 8)


In [10]:
del data
gc.collect()

443

In [11]:
results = []

In [12]:

for model_file in model_files:

    model_path = MODEL_DIR / model_file

    print("\n\n")
    print("=" * 80)
    print(f"MODEL: {model_file}")
    print("=" * 80)


    # --------------------------------------------------------
    # Check file
    # --------------------------------------------------------

    if not model_path.exists():

        print("MODEL NOT FOUND")
        continue


    # --------------------------------------------------------
    # Model size
    # --------------------------------------------------------

    model_size_gb = (
        model_path.stat().st_size
        / (1024 ** 3)
    )

    print(
        f"Model file size: {model_size_gb:.3f} GB"
    )


    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    print("\nLoading...")

    start = time.perf_counter()

    models = joblib.load(model_path)

    load_time = (
        time.perf_counter() - start
    )

    print(
        f"Loaded in {load_time:.3f} seconds"
    )


    # --------------------------------------------------------
    # Check model list
    # --------------------------------------------------------

    if not isinstance(models, (list, tuple)):

        print(
            "ERROR: This file does not contain "
            "a list/tuple of target models."
        )

        print(
            "Loaded type:",
            type(models)
        )

        del models
        gc.collect()

        continue


    print(
        f"Target models found: {len(models)}"
    )


    # ========================================================
    # TEST EACH TARGET
    # ========================================================

    for i, target_name in enumerate(target_names):

        print("\n" + "-" * 80)
        print(
            f"Target {i + 1}/8: {target_name}"
        )
        print("-" * 80)


        if i >= len(models):

            print("No model for this target.")
            continue


        model = models[i]


        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        start = time.perf_counter()

        prediction = model.predict(X_test)

        prediction_time = (
            time.perf_counter() - start
        )


        prediction = np.asarray(
            prediction
        ).reshape(-1)

        actual = Y_test[:, i]


        # ----------------------------------------------------
        # RMSE
        # ----------------------------------------------------

        rmse = np.sqrt(
            mean_squared_error(
                actual,
                prediction
            )
        )


        # ----------------------------------------------------
        # MAE
        # ----------------------------------------------------

        mae = mean_absolute_error(
            actual,
            prediction
        )


        # ----------------------------------------------------
        # R²
        # ----------------------------------------------------

        r2 = r2_score(
            actual,
            prediction
        )


        # ----------------------------------------------------
        # Normalized RMSE
        # ----------------------------------------------------

        actual_std = np.std(actual)

        if actual_std != 0:

            normalized_rmse = (
                rmse / actual_std
            )

        else:

            normalized_rmse = np.nan


        # ----------------------------------------------------
        # Mean Relative Error
        # ----------------------------------------------------

        nonzero = (
            np.abs(actual)
            > np.finfo(float).eps
        )

        if np.any(nonzero):

            mean_relative_error = np.mean(
                np.abs(
                    prediction[nonzero]
                    - actual[nonzero]
                )
                / np.abs(actual[nonzero])
            )

        else:

            mean_relative_error = np.nan


        # ----------------------------------------------------
        # Maximum Absolute Error
        # ----------------------------------------------------

        max_absolute_error = np.max(
            np.abs(
                prediction - actual
            )
        )


        # ----------------------------------------------------
        # Print
        # ----------------------------------------------------

        print(
            f"RMSE            : {rmse:.6e}"
        )

        print(
            f"MAE             : {mae:.6e}"
        )

        print(
            f"R²              : {r2:.8f}"
        )

        print(
            f"Normalized RMSE : {normalized_rmse:.6e}"
        )

        print(
            f"Relative Error  : {mean_relative_error:.6e}"
        )

        print(
            f"Max Error       : {max_absolute_error:.6e}"
        )

        print(
            f"Prediction Time  : {prediction_time:.6f} sec"
        )


        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        results.append({

            "model": model_file,

            "target": target_name,

            "rmse": rmse,

            "mae": mae,

            "r2": r2,

            "normalized_rmse": normalized_rmse,

            "mean_relative_error": mean_relative_error,

            "max_absolute_error": max_absolute_error,

            "model_size_gb": model_size_gb,

            "load_time_sec": load_time,

            "prediction_time_sec": prediction_time,

        })


        # ----------------------------------------------------
        # Free target model + prediction
        # ----------------------------------------------------

        del prediction
        del model

        gc.collect()


    # ========================================================
    # FREE ENTIRE MODEL LIST
    # ========================================================

    del models

    gc.collect()

    print(
        "\nEntire model released from memory."
    )




MODEL: linear_regression.joblib
Model file size: 0.000 GB

Loading...
Loaded in 0.072 seconds
ERROR: This file does not contain a list/tuple of target models.
Loaded type: <class 'sklearn.linear_model._base.LinearRegression'>



MODEL: polynomial_regression_degree_2.joblib
Model file size: 0.000 GB

Loading...
Loaded in 0.011 seconds
ERROR: This file does not contain a list/tuple of target models.
Loaded type: <class 'sklearn.pipeline.Pipeline'>



MODEL: polynomial_regression_degree_3.joblib
Model file size: 0.000 GB

Loading...
Loaded in 0.007 seconds
ERROR: This file does not contain a list/tuple of target models.
Loaded type: <class 'sklearn.pipeline.Pipeline'>



MODEL: polynomial_regression_degree_4.joblib
Model file size: 0.000 GB

Loading...
Loaded in 0.006 seconds
ERROR: This file does not contain a list/tuple of target models.
Loaded type: <class 'sklearn.pipeline.Pipeline'>



MODEL: decision_tree.joblib
Model file size: 0.122 GB

Loading...
Loaded in 0.178 seconds
ERROR:

In [21]:
remaining_models = [
    "linear_regression.joblib",
    "polynomial_regression_degree_2.joblib",
    "polynomial_regression_degree_3.joblib",
    "polynomial_regression_degree_4.joblib",
    "decision_tree.joblib",
    "ExtraTreesRegressor.joblib",
    "RandomForestRegressor.joblib",
]

results_remaining = []


for model_file in remaining_models:

    model_path = MODEL_DIR / model_file

    print("\n\n")
    print("=" * 80)
    print(f"MODEL: {model_file}")
    print("=" * 80)


    if not model_path.exists():

        print("MODEL NOT FOUND")
        continue


    # --------------------------------------------------------
    # FILE SIZE
    # --------------------------------------------------------

    model_size_gb = (
        model_path.stat().st_size
        / (1024 ** 3)
    )

    print(
        f"Model file size: {model_size_gb:.3f} GB"
    )


    # --------------------------------------------------------
    # LOAD
    # --------------------------------------------------------

    print("\nLoading...")

    start = time.perf_counter()

    model = joblib.load(model_path)

    load_time = (
        time.perf_counter() - start
    )

    print(
        f"Loaded in {load_time:.3f} seconds"
    )

    print(
        "Model type:",
        type(model)
    )


    # --------------------------------------------------------
    # PREDICT
    # --------------------------------------------------------

    print("\nPredicting all 8 targets...")

    start = time.perf_counter()

    prediction = model.predict(X_test)

    prediction_time = (
        time.perf_counter() - start
    )

    prediction = np.asarray(prediction)

    print(
        "Prediction shape:",
        prediction.shape
    )

    print(
        f"Prediction time: {prediction_time:.6f} sec"
    )


    # --------------------------------------------------------
    # CHECK
    # --------------------------------------------------------

    if prediction.shape != Y_test.shape:

        print(
            "ERROR: Prediction shape does not match Y_test"
        )

        print(
            "Prediction:",
            prediction.shape
        )

        print(
            "Expected:",
            Y_test.shape
        )

        del prediction
        del model

        gc.collect()

        continue


    # ========================================================
    # EACH TARGET
    # ========================================================

    for i, target_name in enumerate(target_names):

        print("\n" + "-" * 70)
        print(target_name)
        print("-" * 70)


        actual = Y_test[:, i]

        predicted = prediction[:, i]


        # ----------------------------------------------------
        # VALID PREDICTIONS
        # ----------------------------------------------------

        valid = np.isfinite(predicted)

        actual_metric = actual[valid]
        predicted_metric = predicted[valid]


        # ----------------------------------------------------
        # RMSE
        # ----------------------------------------------------

        rmse = np.sqrt(
            mean_squared_error(
                actual_metric,
                predicted_metric
            )
        )


        # ----------------------------------------------------
        # MAE
        # ----------------------------------------------------

        mae = mean_absolute_error(
            actual_metric,
            predicted_metric
        )


        # ----------------------------------------------------
        # R²
        # ----------------------------------------------------

        r2 = r2_score(
            actual_metric,
            predicted_metric
        )


        # ----------------------------------------------------
        # NORMALIZED RMSE
        # ----------------------------------------------------

        std = np.std(actual_metric)

        if std != 0:

            normalized_rmse = (
                rmse / std
            )

        else:

            normalized_rmse = np.nan


        # ----------------------------------------------------
        # RELATIVE ERROR
        # ----------------------------------------------------

        nonzero = (
            np.abs(actual_metric)
            > np.finfo(float).eps
        )


        if np.any(nonzero):

            relative_error = np.mean(
                np.abs(
                    predicted_metric[nonzero]
                    - actual_metric[nonzero]
                )
                /
                np.abs(
                    actual_metric[nonzero]
                )
            )

        else:

            relative_error = np.nan


        # ----------------------------------------------------
        # MAX ERROR
        # ----------------------------------------------------

        max_error = np.max(
            np.abs(
                predicted_metric
                - actual_metric
            )
        )


        # ----------------------------------------------------
        # PRINT
        # ----------------------------------------------------

        print(
            f"RMSE            : {rmse:.6e}"
        )

        print(
            f"MAE             : {mae:.6e}"
        )

        print(
            f"R²              : {r2:.8f}"
        )

        print(
            f"Normalized RMSE : {normalized_rmse:.6e}"
        )

        print(
            f"Relative Error  : {relative_error:.6e}"
        )

        print(
            f"Max Error       : {max_error:.6e}"
        )


        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        results_remaining.append({

            "model": model_file,

            "target": target_name,

            "rmse": rmse,

            "mae": mae,

            "r2": r2,

            "normalized_rmse": normalized_rmse,

            "mean_relative_error": relative_error,

            "max_absolute_error": max_error,

            "model_size_gb": model_size_gb,

            "load_time_sec": load_time,

            "prediction_time_sec": prediction_time,

        })


    # ========================================================
    # FREE EVERYTHING
    # ========================================================

    del prediction
    del model

    gc.collect()




MODEL: linear_regression.joblib
Model file size: 0.000 GB

Loading...
Loaded in 0.001 seconds
Model type: <class 'sklearn.linear_model._base.LinearRegression'>

Predicting all 8 targets...
Prediction shape: (10000, 8)
Prediction time: 0.008195 sec

----------------------------------------------------------------------
position
----------------------------------------------------------------------
RMSE            : 3.411961e+12
MAE             : 9.419380e+11
R²              : -0.02900976
Normalized RMSE : 1.014401e+00
Relative Error  : 2.465365e+04
Max Error       : 2.920115e+13

----------------------------------------------------------------------
velocity
----------------------------------------------------------------------
RMSE            : 8.876338e+07
MAE             : 2.716063e+07
R²              : 0.50420579
Normalized RMSE : 7.041266e-01
Relative Error  : 3.831925e-01
Max Error       : 5.982018e+08

----------------------------------------------------------------------
beta

In [22]:
print("BEST MODEL PER TARGET — R²")
results.extend(results_remaining)
results_df = pd.DataFrame(results)
print(
    "\nTotal evaluation results:",
    len(results_df)
)
results_df.to_csv(
    RESULT_FILE,
    index=False
)

print(
    "\nSaved:",
    RESULT_FILE
)

BEST MODEL PER TARGET — R²

Total evaluation results: 72

Saved: F:\Mayank\ml-research\ml-relativistic-simulator\data\extreme_test_results.csv


In [23]:
print("ALL RESULTS")
print(
    results_df.to_string(index=False)
)
print("BEST MODEL PER TARGET — NORMALIZED RMSE")
best_normalized = (
    results_df
    .sort_values("normalized_rmse")
    .groupby("target", as_index=False)
    .first()
)

best_normalized[
        [
            "target",
            "model",
            "normalized_rmse",
            "r2",
            "mean_relative_error",
        ]
    ].to_string(index=False)

ALL RESULTS
                                model         target         rmse          mae            r2  normalized_rmse  mean_relative_error  max_absolute_error  model_size_gb  load_time_sec  prediction_time_sec
             gradient_boosting.joblib       position 3.410270e+12 9.349058e+11 -2.798999e-02     1.013898e+00         2.559616e+02        2.921318e+13       0.003632       0.103582             0.022580
             gradient_boosting.joblib       velocity 9.319568e+07 2.679813e+07  4.534557e-01     7.392863e-01         1.763179e-01        5.694266e+08       0.003632       0.103582             0.015442
             gradient_boosting.joblib           beta 3.108673e-01 8.938894e-02  4.534557e-01     7.392863e-01         1.763179e-01        1.899403e+00       0.003632       0.103582             0.014666
             gradient_boosting.joblib          gamma 1.793295e+04 5.655366e+02 -9.954519e-04     1.000498e+00         1.256220e-01        1.572485e+06       0.003632       0.103582

'        target                        model  normalized_rmse        r2  mean_relative_error\n          beta     linear_regression.joblib         0.704127  0.504206         3.831925e-01\n         gamma RandomForestRegressor.joblib         1.000498 -0.000995         1.276621e-01\nkinetic_energy     linear_regression.joblib         0.981057  0.037526         1.557000e+06\n      momentum     linear_regression.joblib         0.934652  0.126426         1.766150e+03\n      position     gradient_boosting.joblib         1.013898 -0.027990         2.559616e+02\n   proper_time     gradient_boosting.joblib         1.036379 -0.074082         6.190243e+01\n  total_energy     linear_regression.joblib         0.845115  0.285781         5.509078e+01\n      velocity     linear_regression.joblib         0.704127  0.504206         3.831925e-01'

In [24]:
print("BEST MODEL PER TARGET — R²")
best_r2 = (
    results_df
    .sort_values("r2", ascending=False)
    .groupby("target", as_index=False)
    .first()
)


best_r2[
        [
            "target",
            "model",
            "r2",
            "normalized_rmse",
            "mean_relative_error",
        ]
    ].to_string(index=False)


BEST MODEL PER TARGET — R²


'        target                        model        r2  normalized_rmse  mean_relative_error\n          beta     linear_regression.joblib  0.504206         0.704127         3.831925e-01\n         gamma RandomForestRegressor.joblib -0.000995         1.000498         1.276621e-01\nkinetic_energy     linear_regression.joblib  0.037526         0.981057         1.557000e+06\n      momentum     linear_regression.joblib  0.126426         0.934652         1.766150e+03\n      position     gradient_boosting.joblib -0.027990         1.013898         2.559616e+02\n   proper_time     gradient_boosting.joblib -0.074082         1.036379         6.190243e+01\n  total_energy     linear_regression.joblib  0.285781         0.845115         5.509078e+01\n      velocity     linear_regression.joblib  0.504206         0.704127         3.831925e-01'

### That's actually a very interesting result for your research: excellent interpolation but poor extrapolation.